# P9-IMPL-WILDFIRE-INGEST — NIFC Wildfire Perimeter Ingestion Pipeline

**Author:** @data-engineer | **Date:** 2026-10-22  
**Schema:** `cdm_tmforum.provable_resilience_intelligence`  
**Source:** [NIFC WFIGS Open Data — Interagency Fire Perimeters](https://data-nifc.opendata.arcgis.com/)  
**Refresh:** 2-hour cadence (perimeters update as fires evolve)  
**SID Domain:** Common → GeographicPlace → HazardEvent (wildfire extension)  

## Pipeline Architecture
```
NIFC WFIGS ArcGIS FeatureServer → Fetch+Parse → GeoJSON POLYGON→WKT → stg_nifc_wildfire (Bronze)
                                                                      ↓ MERGE
                                                                 geographic_hazard_event (Gold)
                                                                      ↓ SPATIAL JOIN
                                                                 hazard_span_intersection
```

## Key Transforms
- ArcGIS Epoch ms → TIMESTAMP
- Acres → estimated_radius_km = sqrt(acres * 0.00404686 / π)
- GeoJSON rings → WKT POLYGON/MULTIPOLYGON
- Incident status → PRI severity mapping (Active+low containment=critical, Active+high=high, Contained=medium, Out=low)

## Lakelink Fiber — Provable Resilience Intelligence

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_nifc_wildfire (
  irwin_id              STRING        NOT NULL  COMMENT 'IRWIN (Integrated Reporting of Wildland-Fire Information) unique fire ID',
  perimeter_id          STRING        NOT NULL  COMMENT 'Unique perimeter feature ID from WFIGS. One fire may have multiple perimeters over time.',
  fire_name             STRING        NOT NULL  COMMENT 'Official incident name (e.g. Park Fire, Bridge Fire)',
  fire_type             STRING                  COMMENT 'Incident type: Wildfire | Prescribed Fire | Complex',
  fire_cause            STRING                  COMMENT 'Cause category: Human | Lightning | Unknown | Under Investigation',
  incident_status       STRING                  COMMENT 'Incident status: Active | Contained | Controlled | Out',
  containment_pct       DOUBLE                  COMMENT 'Percent contained (0-100). NULL if not reported.',
  acres_burned          DOUBLE                  COMMENT 'GIS-calculated acres within the fire perimeter',
  discovery_acres       DOUBLE                  COMMENT 'Acres at time of discovery/first report',
  perimeter_wkt         STRING        NOT NULL  COMMENT 'WKT POLYGON or MULTIPOLYGON of fire perimeter boundary. WGS84.',
  centroid_lat          DOUBLE        NOT NULL  COMMENT 'Centroid latitude of fire perimeter for proximity queries',
  centroid_lon          DOUBLE        NOT NULL  COMMENT 'Centroid longitude of fire perimeter for proximity queries',
  estimated_radius_km   DOUBLE                  COMMENT 'Derived: sqrt(acres_burned * 0.00404686 / PI) km. Circular-equivalent radius for buffer zone.',
  discovery_date        TIMESTAMP     NOT NULL  COMMENT 'Fire discovery or first-report datetime (UTC)',
  containment_date      TIMESTAMP               COMMENT 'Date fire reached full containment (UTC). NULL if ongoing.',
  perimeter_date        TIMESTAMP     NOT NULL  COMMENT 'Date/time this perimeter was captured (UTC)',
  personnel_count       INT                     COMMENT 'Number of personnel assigned to the incident',
  structures_threatened INT                     COMMENT 'Number of structures threatened',
  structures_destroyed  INT                     COMMENT 'Number of structures destroyed',
  source_url            STRING        NOT NULL  COMMENT 'NIFC WFIGS feature service URL',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original GeoJSON feature (for audit/replay)'
)
COMMENT 'Staging: NIFC active wildfire perimeters from WFIGS Open Data. One row per perimeter snapshot. Refreshed every 2 hours. Source for geographic_hazard_event wildfire enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT2H',
  'source.api' = 'data-nifc.opendata.arcgis.com/WFIGS',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
)

In [0]:
"""Fetch active/recent wildfire perimeters from NIFC WFIGS ArcGIS FeatureServer and transform to WKT for Lakelink Fiber PRI."""
import requests
import json
import math
from datetime import datetime, timezone, timedelta
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# --- API Configuration ---
# NIFC WFIGS Interagency Fire Perimeters - Current and recent
WFIGS_URL = "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters/FeatureServer/0/query"
SCHEMA = "cdm_tmforum.provable_resilience_intelligence"

# Query: Active/recent fires in CONUS (past 90 days)
params = {
    "where": "1=1",
    "outFields": "*",
    "outSR": "4326",
    "f": "geojson",
    "resultRecordCount": 2000,
    "geometryType": "esriGeometryEnvelope",
    "geometry": "-125,24,-65,50",  # CONUS bounding box
    "inSR": "4326",
    "spatialRel": "esriSpatialRelIntersects"
}

print(f"Fetching NIFC WFIGS wildfire perimeters (CONUS)...")
resp = requests.get(WFIGS_URL, params=params, timeout=120)
resp.raise_for_status()
data = resp.json()
features = data.get("features", [])
print(f"Fetched {len(features)} wildfire perimeter features from NIFC WFIGS")

# --- Helper functions ---
def epoch_ms_to_ts(epoch_ms):
    """Convert ArcGIS epoch milliseconds to datetime (UTC)."""
    if epoch_ms is None:
        return None
    return datetime.fromtimestamp(epoch_ms / 1000.0, tz=timezone.utc)

def compute_estimated_radius(acres):
    """Circular-equivalent radius: sqrt(acres * 0.00404686 / PI) km."""
    if acres is None or acres <= 0:
        return None
    area_km2 = acres * 0.00404686  # acres to km²
    return round(math.sqrt(area_km2 / math.pi), 2)

def rings_to_wkt(geometry):
    """Convert GeoJSON geometry to WKT POLYGON or MULTIPOLYGON."""
    if geometry is None:
        return None
    geom_type = geometry.get("type", "")
    coords = geometry.get("coordinates", [])
    
    if geom_type == "Polygon":
        rings = []
        for ring in coords:
            pts = ", ".join(f"{c[0]} {c[1]}" for c in ring)
            rings.append(f"({pts})")
        return f"POLYGON({', '.join(rings)})"
    elif geom_type == "MultiPolygon":
        polys = []
        for polygon in coords:
            rings = []
            for ring in polygon:
                pts = ", ".join(f"{c[0]} {c[1]}" for c in ring)
                rings.append(f"({pts})")
            polys.append(f"({', '.join(rings)})")
        return f"MULTIPOLYGON({', '.join(polys)})"
    elif geom_type == "Point":
        return f"POINT({coords[0]} {coords[1]})"
    else:
        return None

def compute_centroid(geometry):
    """Compute simple centroid from GeoJSON geometry coordinates."""
    if geometry is None:
        return (None, None)
    geom_type = geometry.get("type", "")
    coords = geometry.get("coordinates", [])
    
    all_pts = []
    if geom_type == "Polygon":
        for ring in coords:
            all_pts.extend(ring)
    elif geom_type == "MultiPolygon":
        for polygon in coords:
            for ring in polygon:
                all_pts.extend(ring)
    elif geom_type == "Point":
        return (coords[1], coords[0])  # lat, lon
    
    if not all_pts:
        return (None, None)
    avg_lon = sum(p[0] for p in all_pts) / len(all_pts)
    avg_lat = sum(p[1] for p in all_pts) / len(all_pts)
    return (round(avg_lat, 6), round(avg_lon, 6))

# --- Transform features ---
now_utc = datetime.now(timezone.utc)
rows = []
skipped = 0
SRC_URL = "https://data-nifc.opendata.arcgis.com/datasets/wfigs-interagency-perimeters"

for f in features:
    props = f.get("properties", {})
    geom = f.get("geometry")
    
    # Extract IRWIN ID and perimeter ID
    irwin_id = props.get("attr_IrwinID") or props.get("irwin_IrwinID") or props.get("poly_IRWINID") or props.get("OBJECTID")
    perimeter_id = str(props.get("OBJECTID", "") or props.get("GlobalID", ""))
    fire_name = props.get("attr_IncidentName") or props.get("irwin_IncidentName") or props.get("poly_IncidentName") or "Unknown"
    
    if not irwin_id or not geom:
        skipped += 1
        continue
    
    irwin_id = str(irwin_id)
    wkt = rings_to_wkt(geom)
    if not wkt:
        skipped += 1
        continue
    
    centroid = compute_centroid(geom)
    if centroid[0] is None:
        skipped += 1
        continue
    
    acres = props.get("attr_GISAcres") or props.get("irwin_DailyAcres") or props.get("poly_GISAcres")
    discovery_acres = props.get("irwin_InitialResponseAcres") or props.get("attr_DiscoveryAcres")
    
    # Date fields (epoch ms from ArcGIS)
    discovery_date = epoch_ms_to_ts(
        props.get("attr_FireDiscoveryDateTime") or props.get("irwin_FireDiscoveryDateTime")
    )
    containment_date = epoch_ms_to_ts(
        props.get("attr_ContainmentDateTime") or props.get("irwin_ContainmentDateTime")
    )
    perimeter_date = epoch_ms_to_ts(
        props.get("attr_PerimeterDateTime") or props.get("poly_DateCurrent")
    )
    if not discovery_date:
        # Fallback: use CreateDate or current time
        discovery_date = epoch_ms_to_ts(props.get("attr_CreateDate")) or now_utc
    if not perimeter_date:
        perimeter_date = now_utc
    
    # Status and containment
    incident_status = props.get("attr_IncidentTypeCategory") or props.get("irwin_IncidentTypeCategory")
    if incident_status and incident_status.lower() in ('rx', 'prescribed fire'):
        fire_type = "Prescribed Fire"
    elif incident_status and 'complex' in str(incident_status).lower():
        fire_type = "Complex"
    else:
        fire_type = "Wildfire"
    
    containment_pct = props.get("attr_PercentContained") or props.get("irwin_PercentContained")
    if containment_pct is not None:
        try:
            containment_pct = float(containment_pct)
        except (ValueError, TypeError):
            containment_pct = None
    
    # Map NIFC fields
    fire_cause = props.get("attr_FireCause") or props.get("irwin_FireCause") or "Unknown"
    
    # Determine incident status from percent contained
    status_raw = props.get("attr_IncidentCommandPostName") or ""
    if containment_pct is not None and containment_pct >= 100:
        inc_status = "Out"
    elif containment_pct is not None and containment_pct >= 80:
        inc_status = "Contained"
    else:
        inc_status = "Active"
    
    rows.append(Row(
        irwin_id=irwin_id,
        perimeter_id=perimeter_id,
        fire_name=str(fire_name),
        fire_type=fire_type,
        fire_cause=str(fire_cause),
        incident_status=inc_status,
        containment_pct=containment_pct,
        acres_burned=float(acres) if acres else None,
        discovery_acres=float(discovery_acres) if discovery_acres else None,
        perimeter_wkt=wkt,
        centroid_lat=centroid[0],
        centroid_lon=centroid[1],
        estimated_radius_km=compute_estimated_radius(float(acres) if acres else None),
        discovery_date=discovery_date,
        containment_date=containment_date,
        perimeter_date=perimeter_date,
        personnel_count=int(props.get("irwin_TotalIncidentPersonnel") or 0) if props.get("irwin_TotalIncidentPersonnel") else None,
        structures_threatened=int(props.get("irwin_ResidencesThreatenedCount") or 0) if props.get("irwin_ResidencesThreatenedCount") else None,
        structures_destroyed=int(props.get("irwin_ResidencesDestroyedCount") or 0) if props.get("irwin_ResidencesDestroyedCount") else None,
        source_url=SRC_URL,
        ingested_at=now_utc,
        raw_json=json.dumps(f)
    ))

print(f"Transformed: {len(rows)} wildfire perimeters ready for upsert ({skipped} skipped)")
if rows:
    acres_vals = [r.acres_burned for r in rows if r.acres_burned]
    radius_vals = [r.estimated_radius_km for r in rows if r.estimated_radius_km]
    status_counts = {}
    for r in rows:
        status_counts[r.incident_status] = status_counts.get(r.incident_status, 0) + 1
    print(f"Acres range: {min(acres_vals):.1f} - {max(acres_vals):.1f}" if acres_vals else "No acres data")
    print(f"Radius range: {min(radius_vals):.2f} - {max(radius_vals):.2f} km" if radius_vals else "No radius data")
    print(f"Status distribution: {status_counts}")

In [0]:
# Upsert transformed wildfire perimeters into staging table
if len(rows) == 0:
    print("⚠️ No wildfire perimeters to upsert. NIFC may have returned empty results.")
else:
    schema = StructType([
        StructField("irwin_id", StringType(), False),
        StructField("perimeter_id", StringType(), False),
        StructField("fire_name", StringType(), False),
        StructField("fire_type", StringType(), True),
        StructField("fire_cause", StringType(), True),
        StructField("incident_status", StringType(), True),
        StructField("containment_pct", DoubleType(), True),
        StructField("acres_burned", DoubleType(), True),
        StructField("discovery_acres", DoubleType(), True),
        StructField("perimeter_wkt", StringType(), False),
        StructField("centroid_lat", DoubleType(), False),
        StructField("centroid_lon", DoubleType(), False),
        StructField("estimated_radius_km", DoubleType(), True),
        StructField("discovery_date", TimestampType(), False),
        StructField("containment_date", TimestampType(), True),
        StructField("perimeter_date", TimestampType(), False),
        StructField("personnel_count", IntegerType(), True),
        StructField("structures_threatened", IntegerType(), True),
        StructField("structures_destroyed", IntegerType(), True),
        StructField("source_url", StringType(), False),
        StructField("ingested_at", TimestampType(), False),
        StructField("raw_json", StringType(), True)
    ])
    
    df = spark.createDataFrame(rows, schema)
    df.createOrReplaceTempView("nifc_batch")
    
    result = spark.sql(f"""
        MERGE INTO {SCHEMA}.stg_nifc_wildfire AS tgt
        USING nifc_batch AS src
        ON tgt.irwin_id = src.irwin_id AND tgt.perimeter_id = src.perimeter_id
        WHEN MATCHED THEN UPDATE SET
            tgt.fire_name = src.fire_name,
            tgt.fire_type = src.fire_type,
            tgt.fire_cause = src.fire_cause,
            tgt.incident_status = src.incident_status,
            tgt.containment_pct = src.containment_pct,
            tgt.acres_burned = src.acres_burned,
            tgt.perimeter_wkt = src.perimeter_wkt,
            tgt.centroid_lat = src.centroid_lat,
            tgt.centroid_lon = src.centroid_lon,
            tgt.estimated_radius_km = src.estimated_radius_km,
            tgt.containment_date = src.containment_date,
            tgt.perimeter_date = src.perimeter_date,
            tgt.personnel_count = src.personnel_count,
            tgt.structures_threatened = src.structures_threatened,
            tgt.structures_destroyed = src.structures_destroyed,
            tgt.ingested_at = src.ingested_at,
            tgt.raw_json = src.raw_json
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    metrics = result.first()
    print(f"✅ Staging MERGE: {metrics['num_inserted_rows']} inserted, {metrics['num_updated_rows']} updated")
    
    stg_count = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.stg_nifc_wildfire").first()[0]
    print(f"Total staging records: {stg_count}")

In [0]:
# Enrich geographic_hazard_event with NIFC wildfire perimeters
# Severity mapping based on incident status + containment %:
#   Active + <30% contained = critical
#   Active + 30-79% contained = high  
#   Contained (80-99%) = medium
#   Out (100%) = low

result = spark.sql(f"""
    MERGE INTO {SCHEMA}.geographic_hazard_event AS gold
    USING (
        SELECT
            CONCAT('HAZ-NIFC-', SUBSTR(MD5(CONCAT(irwin_id, '-', perimeter_id)), 1, 16)) AS hazard_id,
            'wildfire' AS hazard_type,
            CASE
                WHEN incident_status = 'Active' AND (containment_pct IS NULL OR containment_pct < 30) THEN 'critical'
                WHEN incident_status = 'Active' AND containment_pct < 80 THEN 'high'
                WHEN incident_status = 'Contained' THEN 'medium'
                WHEN incident_status = 'Out' THEN 'low'
                ELSE 'medium'
            END AS severity,
            centroid_lat AS latitude,
            centroid_lon AS longitude,
            estimated_radius_km AS radius_km,
            perimeter_wkt AS geom_wkt,
            discovery_date AS event_start,
            containment_date AS event_end,
            'nifc' AS source,
            CONCAT(fire_name, ' (', CAST(ROUND(COALESCE(acres_burned, 0), 0) AS INT), ' acres',
                   CASE WHEN containment_pct IS NOT NULL THEN CONCAT(', ', CAST(ROUND(containment_pct, 0) AS INT), '%% contained') ELSE '' END,
                   ')') AS description,
            CURRENT_TIMESTAMP() AS created_at
        FROM {SCHEMA}.stg_nifc_wildfire
    ) AS stg
    ON gold.hazard_id = stg.hazard_id
    WHEN MATCHED THEN UPDATE SET
        gold.severity = stg.severity,
        gold.latitude = stg.latitude,
        gold.longitude = stg.longitude,
        gold.radius_km = stg.radius_km,
        gold.geom_wkt = stg.geom_wkt,
        gold.event_start = stg.event_start,
        gold.event_end = stg.event_end,
        gold.source = stg.source,
        gold.description = stg.description
    WHEN NOT MATCHED THEN INSERT *
""")

metrics = result.first()
print(f"✅ Gold MERGE complete: {metrics['num_inserted_rows']} inserted, {metrics['num_updated_rows']} updated")

# Verify gold enrichment
gold_nifc = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE source = 'nifc' AND hazard_type = 'wildfire'").first()[0]
gold_all_fire = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE hazard_type = 'wildfire'").first()[0]
print(f"NIFC wildfire events in gold: {gold_nifc}")
print(f"Total wildfire events in gold (NIFC + synthetic): {gold_all_fire}")

In [0]:
# Spatial intersection: find spans within wildfire estimated_radius_km
# Wildfires use POLYGON perimeters. For the intersection pipeline, we use the
# circular-equivalent radius (from acres) centered on the perimeter centroid,
# matching the established Haversine pattern from earthquake/weather/flood pipelines.
# NOTE: Wildfire radii (from acres) are typically MUCH larger than earthquake radii,
# so we expect real NIFC fires to produce actual span intersections.
# proximity_band: direct_impact (<1km) | near_field (1-5km) | buffer_zone (5-radius)

from datetime import datetime, timezone
batch_id = f"batch-nifc-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"

result = spark.sql(f"""
    MERGE INTO {SCHEMA}.hazard_span_intersection AS hsi
    USING (
        SELECT
            SHA2(CONCAT('nifc', wf.irwin_id, '-', wf.perimeter_id, s.span_id), 256) AS intersection_id,
            'nifc' AS hazard_source,
            wf.irwin_id AS hazard_source_id,
            'wildfire' AS hazard_type,
            CASE
                WHEN wf.incident_status = 'Active' AND (wf.containment_pct IS NULL OR wf.containment_pct < 30) THEN 'critical'
                WHEN wf.incident_status = 'Active' AND wf.containment_pct < 80 THEN 'high'
                WHEN wf.incident_status = 'Contained' THEN 'medium'
                WHEN wf.incident_status = 'Out' THEN 'low'
                ELSE 'medium'
            END AS hazard_severity,
            wf.centroid_lat AS hazard_lat,
            wf.centroid_lon AS hazard_lon,
            wf.estimated_radius_km AS hazard_radius_km,
            wf.discovery_date AS hazard_start,
            wf.containment_date AS hazard_end,
            s.span_id,
            s.midpoint_lat AS span_midpoint_lat,
            s.midpoint_lon AS span_midpoint_lon,
            -- Haversine distance in km
            2 * 6371 * ASIN(SQRT(
                POWER(SIN(RADIANS(s.midpoint_lat - wf.centroid_lat) / 2), 2) +
                COS(RADIANS(wf.centroid_lat)) * COS(RADIANS(s.midpoint_lat)) *
                POWER(SIN(RADIANS(s.midpoint_lon - wf.centroid_lon) / 2), 2)
            )) AS distance_km,
            CASE
                WHEN 2 * 6371 * ASIN(SQRT(
                    POWER(SIN(RADIANS(s.midpoint_lat - wf.centroid_lat) / 2), 2) +
                    COS(RADIANS(wf.centroid_lat)) * COS(RADIANS(s.midpoint_lat)) *
                    POWER(SIN(RADIANS(s.midpoint_lon - wf.centroid_lon) / 2), 2)
                )) < 1.0 THEN 'direct_impact'
                WHEN 2 * 6371 * ASIN(SQRT(
                    POWER(SIN(RADIANS(s.midpoint_lat - wf.centroid_lat) / 2), 2) +
                    COS(RADIANS(wf.centroid_lat)) * COS(RADIANS(s.midpoint_lat)) *
                    POWER(SIN(RADIANS(s.midpoint_lon - wf.centroid_lon) / 2), 2)
                )) < 5.0 THEN 'near_field'
                ELSE 'buffer_zone'
            END AS proximity_band,
            CONCAT('HAZ-NIFC-', SUBSTR(MD5(CONCAT(wf.irwin_id, '-', wf.perimeter_id)), 1, 16)) AS hazard_id,
            '{batch_id}' AS batch_id,
            CURRENT_TIMESTAMP() AS computed_at
        FROM {SCHEMA}.stg_nifc_wildfire wf
        CROSS JOIN {SCHEMA}.physical_resource_span s
        WHERE
            wf.estimated_radius_km IS NOT NULL
            AND ABS(s.midpoint_lat - wf.centroid_lat) < (wf.estimated_radius_km / 111.0) * 1.5
            AND ABS(s.midpoint_lon - wf.centroid_lon) < (wf.estimated_radius_km / (111.0 * COS(RADIANS(wf.centroid_lat)))) * 1.5
            AND 2 * 6371 * ASIN(SQRT(
                POWER(SIN(RADIANS(s.midpoint_lat - wf.centroid_lat) / 2), 2) +
                COS(RADIANS(wf.centroid_lat)) * COS(RADIANS(s.midpoint_lat)) *
                POWER(SIN(RADIANS(s.midpoint_lon - wf.centroid_lon) / 2), 2)
            )) <= wf.estimated_radius_km
    ) AS new_ix
    ON hsi.intersection_id = new_ix.intersection_id
    WHEN NOT MATCHED THEN INSERT (
        intersection_id, hazard_source, hazard_source_id, hazard_type, hazard_severity,
        hazard_lat, hazard_lon, hazard_radius_km, hazard_start, hazard_end,
        span_id, span_midpoint_lat, span_midpoint_lon, distance_km, is_within_radius,
        proximity_band, computed_at, batch_id, hazard_id
    ) VALUES (
        new_ix.intersection_id, new_ix.hazard_source, new_ix.hazard_source_id, new_ix.hazard_type, new_ix.hazard_severity,
        new_ix.hazard_lat, new_ix.hazard_lon, new_ix.hazard_radius_km, new_ix.hazard_start, new_ix.hazard_end,
        new_ix.span_id, new_ix.span_midpoint_lat, new_ix.span_midpoint_lon, new_ix.distance_km, TRUE,
        new_ix.proximity_band, new_ix.computed_at, new_ix.batch_id, new_ix.hazard_id
    )
""")

metrics = result.first()
print(f"✅ Hazard-span intersection: {metrics['num_inserted_rows']} new from live NIFC data")
if metrics['num_inserted_rows'] == 0:
    print("   (Wildfires may be in remote areas outside Lakelink Fiber network coverage)")
else:
    # Show top affected spans
    top_spans = spark.sql(f"""
        SELECT hazard_source_id, hazard_severity, COUNT(*) as spans_affected,
               ROUND(MIN(distance_km), 2) as closest_km,
               ROUND(MAX(distance_km), 2) as farthest_km
        FROM {SCHEMA}.hazard_span_intersection
        WHERE batch_id = '{batch_id}'
        GROUP BY hazard_source_id, hazard_severity
        ORDER BY spans_affected DESC
        LIMIT 10
    """)
    print("\nTop wildfire-span intersections:")
    top_spans.show(truncate=False)

total_hsi_nifc = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.hazard_span_intersection WHERE hazard_source = 'nifc'").first()[0]
print(f"Total NIFC wildfire-span intersections: {total_hsi_nifc}")

In [0]:
# Apply COMMENT ON for semantic modeling mandate
spark.sql(f"""
    COMMENT ON TABLE {SCHEMA}.stg_nifc_wildfire IS
    'Staging: NIFC WFIGS active wildfire perimeters. One row per perimeter snapshot per IRWIN fire ID. Refreshed every 2 hours from ArcGIS FeatureServer. POLYGON/MULTIPOLYGON geometry for fire boundary. Source for geographic_hazard_event wildfire enrichment. Lakelink Fiber PRI.'
""")
print("✅ COMMENT ON applied to stg_nifc_wildfire")

# --- Final pipeline run summary ---
stg_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.stg_nifc_wildfire").first()[0]
gold_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event").first()[0]
gold_nifc_wf = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE source = 'nifc' AND hazard_type = 'wildfire'").first()[0]
gold_all_wf = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.geographic_hazard_event WHERE hazard_type = 'wildfire'").first()[0]
hsi_nifc = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.hazard_span_intersection WHERE hazard_source = 'nifc'").first()[0]
hsi_total = spark.sql(f"SELECT COUNT(*) FROM {SCHEMA}.hazard_span_intersection").first()[0]

# Status distribution in staging
status_dist = spark.sql(f"""
    SELECT incident_status, COUNT(*) as cnt,
           ROUND(AVG(acres_burned), 0) as avg_acres,
           ROUND(AVG(estimated_radius_km), 2) as avg_radius_km,
           ROUND(AVG(containment_pct), 0) as avg_pct
    FROM {SCHEMA}.stg_nifc_wildfire
    GROUP BY incident_status
    ORDER BY cnt DESC
""")

print("\n" + "="*70)
print("NIFC WILDFIRE INGESTION PIPELINE — RUN SUMMARY")
print("="*70)
print(f"Staging (stg_nifc_wildfire):        {stg_total:,} perimeters")
print(f"Gold wildfire events (NIFC live):    {gold_nifc_wf:,}")
print(f"Gold wildfire events (all sources):  {gold_all_wf:,}")
print(f"Gold hazard events (all types):      {gold_total:,}")
print(f"NIFC hazard-span intersections:      {hsi_nifc:,}")
print(f"Total hazard-span intersections:     {hsi_total:,}")
print("\nStatus distribution:")
status_dist.show(truncate=False)

# Source breakdown in geographic_hazard_event
print("\nGold geographic_hazard_event by source:")
spark.sql(f"""
    SELECT source, hazard_type, COUNT(*) as cnt
    FROM {SCHEMA}.geographic_hazard_event
    GROUP BY source, hazard_type
    ORDER BY cnt DESC
""").show(truncate=False)

print("\n✅ P9-IMPL-WILDFIRE-INGEST complete")